# ALH_DataFrame_Load_AutoFill_and_Logging — reusable patterns for loading a DataFrame into an Oracle table

A **generic, customer-agnostic** notebook that highlights two reusable helpers layered on top
of the standard AIDP connector write. For the **connector write mechanics themselves**
(external-catalog read/write, `insertInto`, `MERGE`, and the full connector option list), see
the official samples under `data-engineering/ingestion/Read_Write_Oracle_Ecosystem_Connectors/`
in [oracle-samples/oracle-aidp-samples](https://github.com/oracle-samples/oracle-aidp-samples).

**What this demo adds on top of those samples:**

1. **`align_to_target()`** — build only the columns that carry values. The helper reads the
   live table schema, fills any missing column with a typed `NULL`, casts each column to the
   table's type, and orders them to the table — so positional `insertInto` always lines up and
   you never hand-maintain null columns.
2. **Reliable Volume logging + `post_run_cell` hook** — write logs to a local file and copy the
   complete log to a Volume with an explicit close, after each step / on completion / on error /
   after every cell, so logs are captured even if a step fails.

It also shows in-notebook dummy data and surrogate-key generation, and ends with a
**performance-tuning reference** using the documented connector options
(`write.batch.size`, `partition.*`, `fetch.size`).

**Target table:** `ext_private_catalog_aidp_user.coder.sales_demo`

## Prerequisites

Create the target table in **ADW** (run as the schema owner, e.g. `coder`), then **refresh the
`ext_private_catalog_aidp_user` catalog** in AIDP so the table is visible to this notebook.

```sql
CREATE TABLE sales_demo (
    SALE_ID           NUMBER            NOT NULL,
    ORDER_NUMBER      VARCHAR2(50 BYTE),
    CUSTOMER_ID       NUMBER,
    CUSTOMER_NAME     VARCHAR2(100 BYTE),
    PRODUCT_CODE      VARCHAR2(50 BYTE),
    PRODUCT_CATEGORY  VARCHAR2(50 BYTE),
    QUANTITY          NUMBER(10,0),
    UNIT_PRICE        NUMBER(15,2),
    TOTAL_AMOUNT      NUMBER(15,2),
    CURRENCY_CODE     VARCHAR2(3 BYTE),
    SALE_DATE         DATE,
    REGION            VARCHAR2(50 BYTE),
    SALES_REP         VARCHAR2(100 BYTE),
    STATUS            VARCHAR2(20 BYTE),
    W_INSERT_DT       TIMESTAMP(6),
    W_UPDATE_DT       TIMESTAMP(6),
    ETL_BATCH_ID      NUMBER(12,0),
    SOURCE_SYSTEM     VARCHAR2(30 BYTE),
    INTEGRATION_ID    VARCHAR2(100 BYTE),
    COMMENTS          VARCHAR2(4000 BYTE),
    CONSTRAINT PK_SALES_DEMO PRIMARY KEY (SALE_ID)
);
```

Optionally grant access to the catalog identity used to run the notebook:

```sql
GRANT SELECT, INSERT ON sales_demo TO aidp_user;
```

In [33]:
# Cell 1: Import Libraries
import logging
import io
import os
import shutil
import atexit
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, current_timestamp, row_number, monotonically_increasing_id
from pyspark.sql.types import DecimalType, StringType
from pyspark.sql.window import Window

## Reliable logging to a Volume

This pattern writes application logs to a **local file** during the run, then copies the
complete log to the workspace **Volume** using an explicit `open` / `write` / `flush` /
`fsync` / `close` cycle. Writing locally first and copying with an explicit close is a
simple, portable way to make sure the log file is fully written and reliably persisted to
the Volume. The copy runs progressively, on completion, on error (`finally`), on interpreter
exit (`atexit`), and after every cell (`post_run_cell`), so logs are captured even if a step
fails.

In [34]:
# Cell 2: Setup Logging (in-memory stream + Volume .log, commit-on-close safe)
log_stream = io.StringIO()

RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_NAME = f"ALH_SALES_LOAD_DEMO_{RUN_ID}.log"

log_volume_dir = "/Volumes/std_catalog/bronze/logs"     # Volume backing the log folder; Here, logs is the AIDP volume referring a folder in object storage location
os.makedirs(log_volume_dir, exist_ok=True)
log_file_path  = f"{log_volume_dir}/{LOG_NAME}"
local_log_path = f"/tmp/{LOG_NAME}"                       # local POSIX staging

log_formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

logger = logging.getLogger("SALES_LOAD_DEMO")
logger.setLevel(logging.INFO)
logger.propagate = False

# Re-run safe: close (not just clear) any handlers from a previous run.
for _h in logger.handlers[:]:
    try:
        _h.close()
    finally:
        logger.removeHandler(_h)

stream_handler = logging.StreamHandler(log_stream)
stream_handler.setLevel(logging.INFO)
stream_handler.setFormatter(log_formatter)

file_handler = logging.FileHandler(local_log_path, mode="w")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(log_formatter)

logger.addHandler(stream_handler)
logger.addHandler(file_handler)

def write_logs_to_volume():
    """Copy the current local log to the Volume in one open/write/fsync/close cycle
    so the destination file is fully written and flushed. Safe to call repeatedly;
    does not close the logger."""
    try:
        file_handler.flush()
        if os.path.exists(local_log_path):
            with open(local_log_path, "rb") as src, open(log_file_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
                dst.flush()
                try:
                    os.fsync(dst.fileno())
                except OSError:
                    pass
    except Exception as e:
        print(f"WARN: could not write logs to Volume: {e}")

# Backstop 1: commit on kernel exit.
atexit.register(write_logs_to_volume)

# Backstop 2: commit after EVERY cell (success or failure).
def _commit_logs_post_cell(result=None):
    write_logs_to_volume()
try:
    _ip = get_ipython()
    if _ip is not None and not globals().get("_log_hook_registered", False):
        _ip.events.register("post_run_cell", _commit_logs_post_cell)
        globals()["_log_hook_registered"] = True
except Exception:
    pass

logger.info("Logging initialized.")
logger.info(f"Volume log file: {log_file_path}")
write_logs_to_volume()

In [35]:
# Cell 3: Configuration
target_table = "ext_private_catalog_aidp_user.coder.sales_demo"
num_rows     = 22000
batch_id     = int(datetime.now().strftime("%y%m%d%H%M%S"))   # 12-digit numeric batch id

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

logger.info(f"Target table : {target_table}")
logger.info(f"Dummy rows   : {num_rows}")
logger.info(f"ETL batch id : {batch_id}")

## `align_to_target()` — the key reusable helper

`insertInto()` matches columns **by position**, so the DataFrame must provide *every* table
column, in table order. Rather than hand-add null columns and a manual `.select(...)`, this
helper reads the live table schema and: keeps every target column in order, casts each to the
table's datatype, fills any column not in the DataFrame with a typed `NULL`, and drops extras.

In [36]:
# Cell 4: Helper - align a DataFrame to the target table schema
def align_to_target(df: DataFrame, target_table: str) -> DataFrame:
    target_schema = spark.table(target_table).schema
    src = {c.lower(): c for c in df.columns}          # case-insensitive match
    return df.select([
        (col(src[f.name.lower()]) if f.name.lower() in src else lit(None))
            .cast(f.dataType).alias(f.name)
        for f in target_schema.fields
    ])

## Generate dummy SALES data — only the columns that carry values

Note we deliberately **do not** create `INTEGRATION_ID` or `COMMENTS`. `align_to_target()`
fills them with typed `NULL` automatically — demonstrating that you only code the columns
you actually have data for.

In [37]:
# Cell 5: Generate dummy SALES data (value-bearing columns only)
df = (spark.range(1, num_rows + 1).withColumnRenamed("id", "seq")
    .withColumn("ORDER_NUMBER",     F.concat(F.lit("ORD"), (F.lit(100000) + F.col("seq")).cast("string")))
    .withColumn("CUSTOMER_ID",      (F.lit(1000) + (F.col("seq") % 500)))
    .withColumn("CUSTOMER_NAME",    F.concat(F.lit("Customer "), (F.col("seq") % 500).cast("string")))
    .withColumn("PRODUCT_CODE",     F.concat(F.lit("P"), (F.col("seq") % 200).cast("string")))
    .withColumn("PRODUCT_CATEGORY", F.concat(F.lit("CAT"), (F.col("seq") % 5).cast("string")))
    .withColumn("QUANTITY",         (F.lit(1) + (F.col("seq") % 100)))
    .withColumn("UNIT_PRICE",       F.round((F.col("seq") % 100 + F.lit(1)) * F.lit(9.99), 2))
    .withColumn("TOTAL_AMOUNT",     F.round(F.col("QUANTITY") * F.col("UNIT_PRICE"), 2))
    .withColumn("CURRENCY_CODE",    F.lit("USD"))
    .withColumn("SALE_DATE",        F.expr("date_add(DATE'2024-01-01', cast(seq % 365 as int))"))
    .withColumn("REGION",           F.concat(F.lit("Region-"), (F.col("seq") % 4).cast("string")))
    .withColumn("SALES_REP",        F.concat(F.lit("Rep "), (F.col("seq") % 20).cast("string")))
    .withColumn("STATUS",           F.lit("COMPLETED"))
    .withColumn("W_INSERT_DT",      current_timestamp())
    .withColumn("W_UPDATE_DT",      current_timestamp())
    .withColumn("ETL_BATCH_ID",     F.lit(batch_id))
    .withColumn("SOURCE_SYSTEM",    F.lit("DEMO"))
    .drop("seq"))

logger.info(f"Generated dummy sales rows; value columns: {df.columns}")
df.show(5, truncate=False)
write_logs_to_volume()   # commit logs after the data-generation step

+------------+-----------+-------------+------------+----------------+--------+----------+------------+-------------+----------+--------+---------+---------+--------------------------+--------------------------+------------+-------------+
|ORDER_NUMBER|CUSTOMER_ID|CUSTOMER_NAME|PRODUCT_CODE|PRODUCT_CATEGORY|QUANTITY|UNIT_PRICE|TOTAL_AMOUNT|CURRENCY_CODE|SALE_DATE |REGION  |SALES_REP|STATUS   |W_INSERT_DT               |W_UPDATE_DT               |ETL_BATCH_ID|SOURCE_SYSTEM|
+------------+-----------+-------------+------------+----------------+--------+----------+------------+-------------+----------+--------+---------+---------+--------------------------+--------------------------+------------+-------------+
|ORD100001   |1001       |Customer 1   |P1          |CAT1            |2       |19.98     |39.96       |USD          |2024-01-02|Region-1|Rep 1    |COMPLETED|2026-06-15 21:52:43.257541|2026-06-15 21:52:43.257541|260615215233|DEMO         |
|ORD100002   |1002       |Customer 2   |P2  

## Generate the surrogate key, align to the table, and APPEND

`SALE_ID` continues from the current `max()` in the table. `align_to_target()` then fills
`INTEGRATION_ID`/`COMMENTS` as typed `NULL`, casts every column to the table type, and orders
to the table. The load is wrapped in `try/except/finally` so logs are committed to the Volume
as soon as this step completes — success or failure.

In [38]:
# Cell 6: Generate SALE_ID, align to target schema, and APPEND
try:
    max_id_row = spark.table(target_table).selectExpr("max(SALE_ID)").first()[0]
    max_id     = int(max_id_row) if max_id_row is not None else 0
    logger.info(f"Current max SALE_ID: {max_id}. New rows start at {max_id + 1}.")

    window_spec = Window.orderBy(monotonically_increasing_id())
    df = df.withColumn("SALE_ID", (row_number().over(window_spec) + max_id).cast(DecimalType(38, 0)))

    # Fills INTEGRATION_ID + COMMENTS as typed NULL; casts to table types; orders to table.
    df_final = align_to_target(df, target_table)

    logger.info("Final schema being written:")
    for f in df_final.schema.fields:
        logger.info(f"  {f.name}: {f.dataType.simpleString()}")

    (df_final.write
        .option("skip.oos.staging", "true")
        .option("merge.staging.using.username", "true")
        .mode("append")
        .insertInto(target_table))

    logger.info(f"APPEND complete: wrote {df_final.count()} rows to {target_table}.")
except Exception:
    logger.exception("Run failed during load step.")
    raise
finally:
    write_logs_to_volume()

In [39]:
# Cell 7: Final commit + print
write_logs_to_volume()
print(log_stream.getvalue())
print(f"Logs written to: {log_file_path}")

2026-06-15 21:52:28,363 [INFO] Logging initialized.
2026-06-15 21:52:28,364 [INFO] Volume log file: /Volumes/std_catalog/bronze/logs/ALH_SALES_LOAD_DEMO_20260615_215228.log
2026-06-15 21:52:33,816 [INFO] Target table : ext_private_catalog_aidp_user.coder.sales_demo
2026-06-15 21:52:33,816 [INFO] Dummy rows   : 22000
2026-06-15 21:52:33,816 [INFO] ETL batch id : 260615215233
2026-06-15 21:52:43,242 [INFO] Generated dummy sales rows; value columns: ["ORDER_NUMBER", "CUSTOMER_ID", "CUSTOMER_NAME", "PRODUCT_CODE", "PRODUCT_CATEGORY", "QUANTITY", "UNIT_PRICE", "TOTAL_AMOUNT", "CURRENCY_CODE", "SALE_DATE", "REGION", "SALES_REP", "STATUS", "W_INSERT_DT", "W_UPDATE_DT", "ETL_BATCH_ID", "SOURCE_SYSTEM"]
2026-06-15 21:52:50,867 [INFO] Current max SALE_ID: 66400. New rows start at 66401.
2026-06-15 21:52:51,055 [INFO] Final schema being written:
2026-06-15 21:52:51,055 [INFO]   sale_id: decimal(38,10)
2026-06-15 21:52:51,055 [INFO]   order_number: string
2026-06-15 21:52:51,055 [INFO]   customer_

## Appendix — performance tuning with connector options (reference)

For larger loads, the AIDP connector exposes options for **parallelism** and **batching**
(documented in the official `Read_Write_Oracle_Ecosystem_Connectors` samples). For this
200-row demo the basic write above is enough, so the cell below is **off by default**.

| Option | Applies to | Purpose |
| --- | --- | --- |
| `fetch.size` | read | rows fetched per round-trip |
| `write.batch.size` | write | rows written per batch |
| `partition.column` | read / write | numeric or date column for range partitioning |
| `partition.num` | read / write | number of parallel partitions |
| `partition.lower` / `partition.upper` | read / write | range bounds for the partition column |

**Tuned read (parallel, larger fetch):**
```python
df = (spark.read.format("aidataplatform")
    .option("catalog.id", "<external_catalog_id>")
    .option("schema", "coder").option("table", "sales_demo")
    .option("fetch.size", "10000")
    .option("partition.column", "SALE_ID")
    .option("partition.num", "8")
    .option("partition.lower", str(lo)).option("partition.upper", str(hi))
    .load())
```
The runnable, range-partitioned/batched **write** example is in the next cell.

In [40]:
# Appendix (reference): performance-tuned write using documented connector options.
# OFF by default -- running it inserts ANOTHER copy. Set RUN_TUNED_WRITE = True to try it.
RUN_TUNED_WRITE = False

if RUN_TUNED_WRITE:
    # Range bounds for partitioning the write by the numeric key.
    bounds = spark.table(target_table).selectExpr("min(SALE_ID) AS lo", "max(SALE_ID) AS hi").first()
    lo = int(bounds["lo"]) if bounds["lo"] is not None else 0
    hi = int(bounds["hi"]) if bounds["hi"] is not None else (lo + num_rows)

    (df_final.write
        .option("write.mode", "MERGE")
        .option("write.merge.keys", "SALE_ID")
        .option("write.batch.size", "10000")        # rows per write batch
        .option("partition.column", "SALE_ID")      # range-partitioned parallel write
        .option("partition.num", "8")               # number of parallel partitions
        .option("partition.lower", str(lo))
        .option("partition.upper", str(hi))
        .option("skip.oos.staging", "true")
        .option("merge.staging.using.username", "true")
        .mode("append")
        .insertInto(target_table))

    logger.info("Tuned write complete.")
    write_logs_to_volume()
else:
    logger.info("Tuned write skipped (RUN_TUNED_WRITE = False).")
    write_logs_to_volume()